In [ ]:
#| default_exp main

In [ ]:
#| export
import uuid
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path

from dotenv import load_dotenv
from fasthtml.common import *
from monsterui.all import *
from starlette.testclient import TestClient
from string_therapy import create_app

REPO = Path(__file__).resolve().parents[1]
if not (REPO / ".env").exists() and (REPO.parent / ".env").exists():
    REPO = REPO.parent
load_dotenv(REPO / ".env")

_controller = TestClient(create_app())


## Chat UI

Messages go to the in-process `string_therapy` `/controller`. No cole_compass table/chart/analyst paths.


In [ ]:
#| export
class IconKind(Enum):
    INTERFACE = "interface"
    IO = "io"

@dataclass
class Icon:
    id: str
    label: str
    icon: str
    kind: IconKind
    active: bool = False
    extra: dict = field(default_factory=dict)

interface_icons = [
    Icon("chat", "Chat", "message-circle", IconKind.INTERFACE, active=True),
    Icon("graph", "Graph", "share-2", IconKind.INTERFACE),
]

io_icons = [
    Icon("scatter", "Scatter", "chart-scatter", IconKind.IO),
    Icon("heatmap", "Heatmap", "grid-3x3", IconKind.IO),
    Icon("timeseries", "Timeseries", "line-chart", IconKind.IO),
    Icon("table", "Table", "table-2", IconKind.IO),
]

GRAPH_HTML = REPO / "ui" / "graph_network.html"
GRAPH_JSON = REPO / "routes" / "router_graph.json"

_VIEW_IDS = ("chat-view", "graph-view", "scatter-view", "heatmap-view", "timeseries-view", "table-view")
_ACTIVE_OUTLINE_CLASSES = ("ring-2", "ring-primary")

def _show_view_js(view_id: str) -> str:
    """Show one pane inside #main-panel; hide the others."""
    parts = [f"document.getElementById('{v}').classList.add('hidden')" for v in _VIEW_IDS]
    parts.append(f"document.getElementById('{view_id}').classList.remove('hidden')")
    return "; ".join(parts)

def _set_active_icon_js(icon_id: str, *, attr: str = "data-taskbar-icon") -> str:
    """Client-side: outline only the clicked icon in a group."""
    remove = "".join(f"b.classList.remove('{c}');" for c in _ACTIVE_OUTLINE_CLASSES)
    add = "".join(f"t.classList.add('{c}');" for c in _ACTIVE_OUTLINE_CLASSES)
    return (
        f"document.querySelectorAll('[{attr}]').forEach(b=>{{"
        + remove
        + "});"
        + f"const t=document.querySelector('[{attr}=\"{icon_id}\"]');"
        + f"if(t){{{add}}}"
    )

def _icon_btn(ic: Icon, *, color: str = "", **kw):
    outline = " ".join(_ACTIVE_OUTLINE_CLASSES) if ic.active else ""
    return Button(
        Div(
            UkIcon(ic.icon, cls="w-4 h-4"),
            Span(ic.label, cls="text-[10px] leading-tight"),
            cls="flex flex-col items-center gap-0.5",
        ),
        title=ic.label,
        id=f"taskbar-icon-{ic.id}",
        data_taskbar_icon=ic.id,
        cls=f"btn btn-ghost btn-sm h-auto py-1 px-2 {color} {outline}".strip(),
        **kw,
    )

def _io_icon_btn(ic: Icon, **kw):
    """Compact circular button for the chat-area visualization picker."""
    outline = " ".join(_ACTIVE_OUTLINE_CLASSES) if ic.active else ""
    return Button(
        UkIcon(ic.icon, cls="w-3.5 h-3.5"),
        title=ic.label,
        id=f"io-icon-{ic.id}",
        data_io_icon=ic.id,
        type="button",
        cls=f"btn btn-ghost btn-circle btn-xs {outline}".strip(),
        **kw,
    )

def render_icon(ic: Icon):
    if ic.id == "chat":
        return _icon_btn(
            ic,
            onclick="; ".join([
                _set_active_icon_js(ic.id),
                _show_view_js("chat-view"),
            ]),
        )
    if ic.id == "graph":
        return _icon_btn(
            ic,
            onclick="; ".join([
                _set_active_icon_js(ic.id),
                _show_view_js("graph-view"),
            ]),
        )
    return _icon_btn(ic, onclick=_set_active_icon_js(ic.id))

def render_io_icon(ic: Icon):
    """IO / visualization picker icon — opens that viz in the main panel."""
    return _io_icon_btn(
        ic,
        onclick="; ".join([
            _set_active_icon_js(ic.id, attr="data-io-icon"),
            _show_view_js(f"{ic.id}-view"),
        ]),
    )

def taskbar():
    return Div(
        *[render_icon(i) for i in interface_icons],
        Div(cls="flex-1"),
        cls="inline-flex items-center gap-1 p-2 border-b bg-base-200",
    )

def viz_picker():
    """Pill of io_icons perched on the chat input; click opens a visualization."""
    return Div(
        *[render_io_icon(i) for i in io_icons],
        id="viz-picker",
        cls=(
            "absolute left-1/2 -translate-x-1/2 -top-3 z-10 "
            "inline-flex items-center gap-0.5 px-1.5 py-0.5 "
            "rounded-full border bg-base-100 shadow-sm"
        ),
    )

def chat_bubble(role: str, content: str):
    is_user = role == "user"
    return Div(
        Div(
            content,
            cls=f"chat-bubble whitespace-pre-line {'chat-bubble-primary' if is_user else 'chat-bubble-neutral'}",
        ),
        cls=f"chat {'chat-end' if is_user else 'chat-start'} px-4",
    )

def oob_chat_bubble(role: str, content: str):
    """Append a chat bubble via HTMX OOB (wrapper stripped by beforeend)."""
    return Div(
        chat_bubble(role, content),
        hx_swap_oob="beforeend:#chat-messages",
    )

def chat_area():
    """Scrollable chat pane inside #main-panel."""
    return Div(
        Div(id="chat-messages", cls="flex flex-col gap-2 py-4 w-full max-w-2xl mx-auto"),
        id="chat-view",
        cls="flex flex-col flex-1 min-h-0 overflow-y-auto",
    )

def graph_view():
    """Graph pane inside #main-panel (shown when Graph icon is clicked)."""
    return Div(
        Iframe(
            src="/graph",
            title="Router graph",
            cls="w-full h-full border-0",
        ),
        id="graph-view",
        cls="flex-1 min-h-0 overflow-hidden hidden",
    )

def _viz_placeholder(ic: Icon):
    """Placeholder pane for an io visualization until the real panel is wired."""
    return Div(
        Div(
            UkIcon(ic.icon, cls="w-12 h-12 text-base-content/20"),
            P(f"{ic.label} visualization", cls="text-base-content/50 mt-3"),
            cls="flex flex-col items-center justify-center h-full",
        ),
        id=f"{ic.id}-view",
        cls="flex-1 min-h-0 overflow-hidden hidden",
    )

def main_panel():
    """Single content div: chat, graph, or an io visualization fills it."""
    return Div(
        chat_area(),
        graph_view(),
        *[_viz_placeholder(i) for i in io_icons],
        id="main-panel",
        cls="flex flex-col flex-1 min-h-0 overflow-hidden",
    )

def input_bar():
    return Div(
        Form(
            Div(
                viz_picker(),
                Textarea(
                    placeholder="Type a message…",
                    name="message",
                    rows=2,
                    cls="textarea textarea-bordered w-full resize-none pt-6 pr-14 focus:outline-none",
                    id="chat-input",
                    onkeydown="if(event.key==='Enter'&&!event.shiftKey){event.preventDefault();this.form.requestSubmit()}",
                    hx_on_input="""
                        let btn = document.getElementById('submit-btn');
                        if (this.value.trim().length > 0) {
                            btn.classList.remove('hidden');
                        } else {
                            btn.classList.add('hidden');
                        }
                        """,
                ),
                Button(
                    UkIcon("arrow-up", cls="w-5 h-5"),
                    id="submit-btn",
                    cls="btn btn-primary btn-circle btn-sm absolute right-3 bottom-3 hidden",
                ),
                cls="relative w-full",
            ),
            hx_post="/chat/send",
            hx_target="#htmx-sink",
            hx_swap="none",
            hx_on__after_request="document.getElementById('chat-input').value=''; document.getElementById('submit-btn').classList.add('hidden')",
            cls="w-full max-w-2xl mx-auto",
        ),
        id="input-bar",
        cls="p-3 pt-5 border-t bg-base-100",
    )

app, rt = fast_app(hdrs=Theme.blue.headers())

@rt("/graph")
def graph():
    """Serve graph_network.html with router_graph.json injected."""
    import json as _json
    html = GRAPH_HTML.read_text(encoding="utf-8")
    data = _json.loads(GRAPH_JSON.read_text(encoding="utf-8"))
    marker = '<script id="graph-data" type="application/json">null</script>'
    payload = f'<script id="graph-data" type="application/json">{_json.dumps(data)}</script>'
    if marker not in html:
        raise RuntimeError("graph_network.html missing #graph-data injection point")
    return Response(html.replace(marker, payload, 1), media_type="text/html")

@rt("/")
def get(session):
    if "session_id" not in session:
        session["session_id"] = str(uuid.uuid4())

    return Div(
        Div(taskbar(), cls="flex justify-center border-b"),
        main_panel(),
        input_bar(),
        Div(id="htmx-sink", cls="hidden", aria_hidden="true"),
        cls="flex flex-col h-screen",
    )

@rt("/chat/send", methods=["POST"])
async def chat_send(message: str, session):
    user_msg = message.strip()
    if not user_msg:
        return ""

    sid = session.get("session_id", "").strip()
    payload: dict = {"message": user_msg, "use_history": True, "response_format": "json"}
    if sid:
        payload["session_id"] = sid

    r = _controller.post("/controller", json=payload)
    try:
        data = r.json()
    except Exception:  # noqa: BLE001
        data = {"detail": r.text[:2000]}

    if r.status_code >= 400:
        reply = data.get("detail") or data.get("error") or f"Controller error HTTP {r.status_code}"
    else:
        reply = data.get("response") or data.get("detail") or "No response"
        if not isinstance(reply, str):
            import json as _json
            reply = _json.dumps(reply, indent=2)

    new_sid = (data.get("session_id") or sid or "").strip()
    if new_sid:
        session["session_id"] = new_sid

    return (
        oob_chat_bubble("user", user_msg),
        oob_chat_bubble("assistant", reply),
        Script(
            "requestAnimationFrame(()=>{const c=document.getElementById('chat-messages');"
            "if(c)c.scrollTop=c.scrollHeight;})"
        ),
    )


if __name__ == "__main__":
    serve(appname="string_therapy_for_material_science.main", port=4545)


## Router graph

Directed force layout of `routes/router_graph.json`, rendered from `ui/graph_network.html`.
Drag nodes, scroll to zoom, hover for endpoint details.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

from IPython.display import IFrame, display

repo = Path.cwd()
if not (repo / "routes" / "router_graph.json").exists() and (repo.parent / "routes" / "router_graph.json").exists():
    repo = repo.parent

graph_path = repo / "routes" / "router_graph.json"
html_path = repo / "ui" / "graph_network.html"

graph = json.loads(graph_path.read_text(encoding="utf-8"))
html = html_path.read_text(encoding="utf-8")

# Inject graph JSON so the viz works inside the notebook without a static file server.
marker = '<script id="graph-data" type="application/json">null</script>'
payload = f'<script id="graph-data" type="application/json">{json.dumps(graph)}</script>'
if marker not in html:
    raise RuntimeError("graph_network.html missing #graph-data injection point")
html = html.replace(marker, payload, 1)

print(f"nodes={len(graph['nodes'])} edges={len(graph['edges'])}")
display(IFrame(srcdoc=html, width="100%", height=560))
